Notebook: 01_dataset_audit.ipynb

Purpose: Inventory datasets and export standardized record metadata.

Inputs:
- raw dataset files in data/*

Outputs:
- inventory.csv
- clinical_context.parquet

Notes:
- Must produce record-level metadata exact to DATA_CONTRACT.md.

# 01 — Dataset Audit and Inventory

Collect dataset metadata, verify sampling information, and export a canonical record inventory for downstream evidence pipelines.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
sys.path.insert(0, str(Path.cwd().parent / 'src'))
from ecg_analytics.datasets import PTBXLDataset, LUDBDataset, NSTDBDataset, INCARTDataset, QTDBDataset

root_dir = Path.cwd().parent
artifacts_dir = root_dir / 'artifacts'
artifacts_dir.mkdir(exist_ok=True)
data_dir = root_dir / 'data'

adapters = {
    'ptbxl': PTBXLDataset,
    'ludb': LUDBDataset,
    'nstdb': NSTDBDataset,
    'incart': INCARTDataset,
    'qtdb': QTDBDataset,
}

inventory_rows = []
clinical_rows = []

for dataset_name, adapter_cls in adapters.items():
    dataset_path = data_dir / dataset_name
    if not dataset_path.exists():
        continue
    adapter = adapter_cls(data_dir=data_dir)
    try:
        record_names = adapter.list_records()
    except Exception as exc:
        print(f'Warning: {dataset_name} listing failed: {exc}')
        continue

    for record_name in record_names:
        try:
            record = adapter.load_record(record_name)
        except Exception as exc:
            print(f'Warning: failed loading {dataset_name}/{record_name}: {exc}')
            continue

        record_id = f'{dataset_name}/{record_name}'
        lead_names = [str(name).lower() for name in getattr(record, 'lead_names', [])] if getattr(record, 'lead_names', None) else []
        sampling_rate = float(getattr(record, 'fs', np.nan))
        duration_seconds = float(getattr(record, 'duration_s', np.nan))
        time_resolution_ms = float(1000.0 / sampling_rate) if sampling_rate and sampling_rate > 0 else np.nan
        annotations = getattr(record, 'annotations', []) or []
        annotation_source = 'manual' if annotations else 'none'
        has_manual_t_end = any(
            getattr(getattr(ann, 'symbol', None), 'lower', lambda: '')() in {'t', ')'}
            or getattr(getattr(ann, 'label', None), 'lower', lambda: '')().startswith('t-end')
            for ann in annotations
        )

        inventory_rows.append({
            'record_id': record_id,
            'dataset_name': dataset_name,
            'sampling_rate': sampling_rate,
            'time_resolution_ms': time_resolution_ms,
            'duration_seconds': duration_seconds,
            'num_leads': int(getattr(record, 'n_leads', len(lead_names))),
            'lead_names': lead_names,
            'annotation_source': annotation_source,
            'has_manual_t_end': bool(has_manual_t_end),
        })
        clinical_rows.append({
            'record_id': record_id,
            'arrhythmia_flag': bool(dataset_name == 'incart'),
            'diagnostic_class': record.metadata.get('diagnostic_class', np.nan) if getattr(record, 'metadata', None) else np.nan,
            'dataset_origin': dataset_name,
        })

inventory = pd.DataFrame(inventory_rows)
clinical_context = pd.DataFrame(clinical_rows)
expected_inventory = {
    'record_id', 'dataset_name', 'sampling_rate', 'time_resolution_ms',
    'duration_seconds', 'num_leads', 'lead_names', 'annotation_source',
    'has_manual_t_end',
}
assert expected_inventory.issubset(set(inventory.columns)), 'inventory missing required columns'
assert inventory['record_id'].is_unique, 'record_id must be unique'
expected_clinical = {'record_id', 'arrhythmia_flag', 'diagnostic_class', 'dataset_origin'}
assert expected_clinical.issubset(set(clinical_context.columns)), 'clinical_context missing required columns'

inventory.to_csv(artifacts_dir / 'inventory.csv', index=False)
clinical_context.to_parquet(artifacts_dir / 'clinical_context.parquet', index=False)
print('Wrote inventory and clinical_context artifacts')
